# Feature Stream Ablation Study

Runs the full Cycle 1 prediction pipeline under seven feature configurations
and compares results side-by-side.

| Configuration | Feature streams |
|---|---|
| **Tabular only** | Chemical composition (161 features) |
| **Morph only** | Morphological descriptors (33 features) |
| **Image only** | DINOv2 ViT-B/14 embeddings (768 features) |
| **Tabular + Morph** | Tabular + morphological (194 features) |
| **Tabular + Image** | Tabular + DINOv2 (929 features) |
| **Morph + Image** | Morphological + DINOv2 (801 features) |
| **Tabular + Morph + Image** | All three streams (962 features) |

Each configuration is evaluated with:
- RepeatedKFold cross-validation (5×10) for a stable R² estimate
- A held-out test split (15%) for final test R², MAE, RMSE
- Per-target breakdown (HoldingTemp, HoldingTime)
- Feature importance bar chart (RF importances, streams colour-coded)
- Predicted-vs-actual scatter plots for the best model per configuration

In [ ]:
import os, sys, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_val_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    AdaBoostRegressor, ExtraTreesRegressor,
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, make_scorer

sys.path.insert(0, os.path.abspath('..'))

from src.config import Config, PreprocessingConfig, MissingDataConfig, ScalingConfig, EncodingConfig
from src.preprocessing import FeaturePreprocessor
from src.extraction import MorphologicalExtractor, MorphologyConfig
from src.features import FeaturePipeline
from src.column_sanitizer import sanitize_dataframe

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
pd.set_option('display.max_columns', 50)

## 1. Load Data

In [ ]:
df_raw = pd.read_csv('../datasets/metadata_latest.csv', header=1)
df_raw = sanitize_dataframe(df_raw)
df = df_raw[df_raw['c'].notna()].copy().reset_index(drop=True)
print(f'Usable samples: {len(df)}')

# Cycle 1 targets
c1_temp_col = 'cycle1_holdingtemp_degc'
c1_time_col = 'cycle1_holdingtime_min'
target_columns = [c1_temp_col, c1_time_col]

c1_mask = df[c1_temp_col].notna() & df[c1_time_col].notna()
df_c1 = df[c1_mask].copy().reset_index(drop=True)
df_c1['temp_time_ratio'] = df_c1[c1_temp_col] / (df_c1[c1_time_col] + 1e-6)

Y_c1 = df_c1[target_columns].values.astype(np.float64)
print(f'Cycle 1 subset: {len(df_c1)} samples')
for i, col in enumerate(target_columns):
    v = Y_c1[:, i]
    print(f'  {col}: [{v.min():.1f}, {v.max():.1f}]  mean={v.mean():.1f}  std={v.std():.1f}')

## 2. Train / Val / Test Split

Split is performed **before** fitting the preprocessor to prevent MICE/imputer leakage.
Same `random_state=42` as the main demo notebook so results are directly comparable.

In [ ]:
idx = np.arange(len(df_c1))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=42)
idx_train, idx_val     = train_test_split(idx_trainval, test_size=0.15, random_state=42)

Y_train = Y_c1[idx_train]
Y_val   = Y_c1[idx_val]
Y_test  = Y_c1[idx_test]

print(f'Split — train: {len(idx_train)}, val: {len(idx_val)}, test: {len(idx_test)}')

## 3. Tabular Feature Preprocessing

In [ ]:
TARGET_COLUMNS_TO_EXCLUDE = [
    'cycle1_holdingtemp_degc', 'cycle1_holdingtime_min',
    'cycle2_holdingtemp_degc', 'cycle2_holdingtime_min',
    'cycle3_holdingtemp_degc', 'cycle3_holdingtime_min',
    'cycle4_holdingtemp_degc', 'cycle4_holdingtime_min',
]
feature_columns = [c for c in df_c1.columns if c not in TARGET_COLUMNS_TO_EXCLUDE]

preproc_config = PreprocessingConfig(
    missing_data=MissingDataConfig(
        column_drop_threshold=0.95,
        row_fill_threshold=0.50,
        numeric_fill_strategy='median',
        categorical_fill_strategy='mode',
        mice_max_iter=10,
    ),
    scaling=ScalingConfig(method='standard', enabled=True),
    encoding=EncodingConfig(categorical='onehot', max_categories=50),
)

MICE_COLUMNS      = ['cr', 'mo', 's', 'p', 'ni', 'al']
INDICATOR_COLUMNS = ['ti', 'nb', 'v']
COLUMN_TYPE_OVERRIDES = {
    'num_cycles': 'categorical', 'cycle1_crate_degc_s': 'categorical',
    'cycle1_rtemp': 'categorical', 'cycle1_qtemp': 'categorical',
    'cycle2_rtemp_degc': 'categorical', 'cycle3_rtemp': 'categorical',
    'cycle3_qtemp': 'categorical', 'cycle4_qtemp': 'categorical',
    'heat_treatment_type': 'categorical', 'id': 'unique_string',
}

active_overrides = {k: v for k, v in COLUMN_TYPE_OVERRIDES.items() if k in feature_columns}
mice_cols        = [c for c in MICE_COLUMNS      if c in feature_columns]
indicator_cols   = [c for c in INDICATOR_COLUMNS if c in feature_columns]

df_train_c1 = df_c1.iloc[idx_train].reset_index(drop=True)
df_val_c1   = df_c1.iloc[idx_val].reset_index(drop=True)
df_test_c1  = df_c1.iloc[idx_test].reset_index(drop=True)

preprocessor = FeaturePreprocessor(
    preproc_config,
    column_types=active_overrides,
    mice_columns=mice_cols,
    indicator_columns=indicator_cols,
)
print('Fitting tabular preprocessor on training split...')
X_tab_train = preprocessor.fit_transform(df_train_c1[feature_columns].copy())
X_tab_val   = preprocessor.transform(df_val_c1[feature_columns].copy())
X_tab_test  = preprocessor.transform(df_test_c1[feature_columns].copy())

tab_feature_names = list(preprocessor.get_feature_names())
print(f'Tabular feature dim: {X_tab_train.shape[1]}')

## 4. Load Morphological Features

In [ ]:
FEATURES_DIR = os.path.abspath('../features')
DATA_DIR     = os.path.abspath('../data')
TEMP_DIR     = os.path.abspath('../data/temp_images')

fp = FeaturePipeline(data_dir=DATA_DIR, temp_dir=TEMP_DIR, features_dir=FEATURES_DIR)

X_morph_train = X_morph_val = X_morph_test = None
_X_morph_all = fp.load_morph_features(df_c1['id'])

if _X_morph_all is not None:
    X_morph_train = _X_morph_all[idx_train]
    X_morph_val   = _X_morph_all[idx_val]
    X_morph_test  = _X_morph_all[idx_test]
    morph_feature_names = MorphologicalExtractor.get_feature_names()

    morph_preproc_config = PreprocessingConfig(
        missing_data=MissingDataConfig(numeric_fill_strategy='median'),
        scaling=ScalingConfig(method='standard', enabled=True),
        encoding=EncodingConfig(categorical='onehot'),
    )
    morph_preprocessor = FeaturePreprocessor(morph_preproc_config)
    df_morph_train = pd.DataFrame(X_morph_train, columns=morph_feature_names)
    df_morph_val   = pd.DataFrame(X_morph_val,   columns=morph_feature_names)
    df_morph_test  = pd.DataFrame(X_morph_test,  columns=morph_feature_names)
    X_morph_train = morph_preprocessor.fit_transform(df_morph_train)
    X_morph_val   = morph_preprocessor.transform(df_morph_val)
    X_morph_test  = morph_preprocessor.transform(df_morph_test)

    print(f'Morphological features loaded: {_X_morph_all.shape}')
    print(f'Splits — train={X_morph_train.shape}, val={X_morph_val.shape}, test={X_morph_test.shape}')
else:
    morph_feature_names = []
    print('WARNING: Morph cache not found. Run prepare_features.ipynb first.')

## 5. Load Image Features (DINOv2 ViT-B/14)

Uses the pre-built cache from `prepare_features.ipynb`.
Change `BACKBONE` to any cached backbone (e.g. `dinov2_vitl14`, `resnet50`, `vgg16`).

In [ ]:
BACKBONE = os.environ.get('BACKBONE', 'dinov2_vitb14')

X_img_train = X_img_val = X_img_test = None
img_feature_names = []

_X_img_all = fp.load_image_features(BACKBONE, df_c1['id'])

if _X_img_all is not None:
    X_img_train = _X_img_all[idx_train].astype(np.float64)
    X_img_val   = _X_img_all[idx_val].astype(np.float64)
    X_img_test  = _X_img_all[idx_test].astype(np.float64)
    img_feature_names = [f'img_{i}' for i in range(_X_img_all.shape[1])]
    print(f'Image features loaded: backbone={BACKBONE}  dim={_X_img_all.shape[1]}')
    print(f'Splits — train={X_img_train.shape}, val={X_img_val.shape}, test={X_img_test.shape}')
else:
    print(f'WARNING: No cache for backbone={BACKBONE!r}. Run prepare_features.ipynb first.')
    print('Image configurations will be skipped.')

## 6. Build Feature Matrices per Configuration

In [ ]:
# Stream colour palette
COLOR_TAB   = '#2196F3'  # blue
COLOR_MORPH = '#FF9800'  # orange
COLOR_IMG   = '#9C27B0'  # purple
COLOR_COMBO = '#4CAF50'  # green

configs = {}

# ── Single-stream ────────────────────────────────────────────────────────────
configs['Tabular only'] = {
    'X_train': X_tab_train, 'X_val': X_tab_val, 'X_test': X_tab_test,
    'feat_names': tab_feature_names, 'color': COLOR_TAB,
    'streams': ['tab'],
}

if X_morph_train is not None:
    configs['Morph only'] = {
        'X_train': X_morph_train, 'X_val': X_morph_val, 'X_test': X_morph_test,
        'feat_names': morph_feature_names, 'color': COLOR_MORPH,
        'streams': ['morph'],
    }

if X_img_train is not None:
    configs['Image only'] = {
        'X_train': X_img_train, 'X_val': X_img_val, 'X_test': X_img_test,
        'feat_names': img_feature_names, 'color': COLOR_IMG,
        'streams': ['img'],
    }

# ── Two-stream combinations ───────────────────────────────────────────────────
if X_morph_train is not None:
    configs['Tabular + Morph'] = {
        'X_train': np.concatenate([X_morph_train, X_tab_train], axis=1),
        'X_val':   np.concatenate([X_morph_val,   X_tab_val],   axis=1),
        'X_test':  np.concatenate([X_morph_test,  X_tab_test],  axis=1),
        'feat_names': morph_feature_names + tab_feature_names,
        'color': COLOR_COMBO, 'streams': ['morph', 'tab'],
    }

if X_img_train is not None:
    configs['Tabular + Image'] = {
        'X_train': np.concatenate([X_img_train, X_tab_train], axis=1),
        'X_val':   np.concatenate([X_img_val,   X_tab_val],   axis=1),
        'X_test':  np.concatenate([X_img_test,  X_tab_test],  axis=1),
        'feat_names': img_feature_names + tab_feature_names,
        'color': '#00BCD4', 'streams': ['img', 'tab'],
    }

if X_morph_train is not None and X_img_train is not None:
    configs['Morph + Image'] = {
        'X_train': np.concatenate([X_img_train, X_morph_train], axis=1),
        'X_val':   np.concatenate([X_img_val,   X_morph_val],   axis=1),
        'X_test':  np.concatenate([X_img_test,  X_morph_test],  axis=1),
        'feat_names': img_feature_names + morph_feature_names,
        'color': '#FF5722', 'streams': ['img', 'morph'],
    }

# ── All three streams ─────────────────────────────────────────────────────────
if X_morph_train is not None and X_img_train is not None:
    configs['Tab + Morph + Image'] = {
        'X_train': np.concatenate([X_img_train, X_morph_train, X_tab_train], axis=1),
        'X_val':   np.concatenate([X_img_val,   X_morph_val,   X_tab_val],   axis=1),
        'X_test':  np.concatenate([X_img_test,  X_morph_test,  X_tab_test],  axis=1),
        'feat_names': img_feature_names + morph_feature_names + tab_feature_names,
        'color': '#607D8B', 'streams': ['img', 'morph', 'tab'],
    }

print(f'Backbone: {BACKBONE}')
print('\nFeature configurations:')
for name, cfg in configs.items():
    print(f'  {name:<24} {cfg["X_train"].shape[1]:>4} features  streams={cfg["streams"]}')

## 7. Model Definitions

Same four models used in `microstructure_demo.ipynb`.

In [ ]:
RF_KW  = {'n_estimators': 200, 'max_features': 'sqrt', 'min_samples_leaf': 3}
ET_KW  = {'n_estimators': 200, 'max_features': 'sqrt', 'min_samples_leaf': 3}
GBR_KW = {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3,
           'subsample': 0.8, 'min_samples_leaf': 5}
ABR_KW = {'n_estimators': 100, 'learning_rate': 1.0}

def make_models():
    return {
        'GBR':        MultiOutputRegressor(GradientBoostingRegressor(**GBR_KW, random_state=42)),
        'ExtraTrees': ExtraTreesRegressor(**ET_KW, random_state=42, n_jobs=-1),
        'RF':         RandomForestRegressor(**RF_KW, random_state=42, n_jobs=-1),
        'ABR':        MultiOutputRegressor(AdaBoostRegressor(
                          estimator=DecisionTreeRegressor(max_depth=3),
                          **ABR_KW, random_state=42)),
    }

_multi_r2 = make_scorer(r2_score, multioutput='uniform_average')
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

print('Models:', list(make_models().keys()))
print('CV: RepeatedKFold(n_splits=5, n_repeats=10)')

## 8. Run Ablation — CV + Test Evaluation

For each configuration: cross-validate all four models, then fit on the train split
and evaluate on the held-out test set.

In [ ]:
results = {}

# Full arrays for CV (no held-out test leakage — CV folds handle that)
X_all = {
    'Tabular only':        np.vstack([X_tab_train,   X_tab_val,   X_tab_test]),
}
if X_morph_train is not None:
    X_all['Morph only'] = np.vstack([X_morph_train, X_morph_val, X_morph_test])
    X_all['Tabular + Morph'] = np.concatenate([
        np.vstack([X_morph_train, X_morph_val, X_morph_test]),
        np.vstack([X_tab_train,   X_tab_val,   X_tab_test]),
    ], axis=1)
if X_img_train is not None:
    X_all['Image only'] = np.vstack([X_img_train, X_img_val, X_img_test])
    X_all['Tabular + Image'] = np.concatenate([
        np.vstack([X_img_train, X_img_val, X_img_test]),
        np.vstack([X_tab_train, X_tab_val, X_tab_test]),
    ], axis=1)
if X_morph_train is not None and X_img_train is not None:
    X_all['Morph + Image'] = np.concatenate([
        np.vstack([X_img_train,   X_img_val,   X_img_test]),
        np.vstack([X_morph_train, X_morph_val, X_morph_test]),
    ], axis=1)
    X_all['Tab + Morph + Image'] = np.concatenate([
        np.vstack([X_img_train,   X_img_val,   X_img_test]),
        np.vstack([X_morph_train, X_morph_val, X_morph_test]),
        np.vstack([X_tab_train,   X_tab_val,   X_tab_test]),
    ], axis=1)

Y_all = np.vstack([Y_train, Y_val, Y_test])

for cfg_name, cfg in configs.items():
    print(f'\n{"-" * 65}')
    print(f'Configuration: {cfg_name}  ({cfg["X_train"].shape[1]} features)')
    print(f'{"-" * 65}')
    results[cfg_name] = {}

    models = make_models()
    X_cv = X_all[cfg_name]

    for model_name, model in models.items():
        cv_scores = cross_val_score(model, X_cv, Y_all, cv=cv,
                                    scoring=_multi_r2, n_jobs=-1)

        m = copy.deepcopy(model)
        m.fit(cfg['X_train'], Y_train)
        Y_pred = m.predict(cfg['X_test'])
        if Y_pred.ndim == 1:
            Y_pred = Y_pred.reshape(-1, 1)

        test_r2   = r2_score(Y_test, Y_pred, multioutput='uniform_average')
        test_mae  = mean_absolute_error(Y_test, Y_pred, multioutput='uniform_average')
        test_rmse = float(np.sqrt(mean_squared_error(Y_test, Y_pred,
                                                      multioutput='uniform_average')))
        per_target = [r2_score(Y_test[:, i], Y_pred[:, i])
                      for i in range(len(target_columns))]

        results[cfg_name][model_name] = {
            'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
            'test_r2': test_r2, 'test_mae': test_mae, 'test_rmse': test_rmse,
            'per_target': per_target, 'Y_pred': Y_pred, 'model': m,
        }
        print(f'  {model_name:<12} CV R²={cv_scores.mean():.4f}±{cv_scores.std():.4f}  '
              f'test R²={test_r2:.4f}  MAE={test_mae:.2f}')

print('\nDone.')

## 9. Summary Table

In [ ]:
rows = []
for cfg_name, model_results in results.items():
    for model_name, m in model_results.items():
        pt = '  '.join(f'{v:.3f}' for v in m['per_target'])
        rows.append({
            'Configuration': cfg_name,
            'Model':         model_name,
            'CV R² (mean)':  round(m['cv_mean'], 4),
            'CV R² (std)':   round(m['cv_std'],  4),
            'Test R²':       round(m['test_r2'],  4),
            'Test MAE':      round(m['test_mae'],  3),
            'Test RMSE':     round(m['test_rmse'], 3),
            'Per-target R²': pt,
        })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(['Configuration', 'Test R²'], ascending=[True, False])
print(summary_df.to_string(index=False))

## 10. CV R² Comparison — All Models × All Configurations

In [ ]:
cfg_names   = list(results.keys())
model_names = list(next(iter(results.values())).keys())
n_cfgs      = len(cfg_names)
n_models    = len(model_names)
colors      = [configs[c]['color'] for c in cfg_names]

x     = np.arange(n_models)
width = 0.12

fig, ax = plt.subplots(figsize=(14, 5))
for i, (cfg_name, color) in enumerate(zip(cfg_names, colors)):
    means = [results[cfg_name][m]['cv_mean'] for m in model_names]
    stds  = [results[cfg_name][m]['cv_std']  for m in model_names]
    ax.bar(x + i * width, means, width, label=cfg_name, color=color, alpha=0.85,
           yerr=stds, capsize=3, error_kw={'linewidth': 1.0})

ax.set_xticks(x + width * (n_cfgs - 1) / 2)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('CV R² (mean ± std)', fontsize=11)
ax.set_title('Cross-validation R² by Feature Configuration and Model\n'
             '(RepeatedKFold 5×10, uniform average across targets)', fontsize=12)
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(-0.35, 1.05)
ax.axhline(0, color='grey', linewidth=0.5)
plt.tight_layout()
plt.savefig('../runs/ablation_cv_r2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: runs/ablation_cv_r2.png')

## 11. Test R² Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for i, (cfg_name, color) in enumerate(zip(cfg_names, colors)):
    test_r2s = [results[cfg_name][m]['test_r2'] for m in model_names]
    ax.bar(x + i * width, test_r2s, width, label=cfg_name, color=color, alpha=0.85)

ax.set_xticks(x + width * (n_cfgs - 1) / 2)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('Test R²', fontsize=11)
ax.set_title('Held-out Test R² by Feature Configuration and Model\n'
             '(15% hold-out, uniform average across targets)', fontsize=12)
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(-0.15, 1.05)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.savefig('../runs/ablation_test_r2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: runs/ablation_test_r2.png')

## 12. Delta Heatmap — Stream Contributions

Delta = test R² of each configuration minus Tabular-only baseline.

In [ ]:
baseline = 'Tabular only'
non_baseline = [c for c in cfg_names if c != baseline]

delta_data = {}
for cfg_name in non_baseline:
    delta_data[cfg_name] = {}
    for model_name in model_names:
        delta_data[cfg_name][model_name] = (
            results[cfg_name][model_name]['test_r2']
            - results[baseline][model_name]['test_r2']
        )

delta_df = pd.DataFrame(delta_data, index=model_names).T

fig, ax = plt.subplots(figsize=(9, 0.7 * len(non_baseline) + 1.5))
sns.heatmap(delta_df, annot=True, fmt='+.4f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'ΔTest R² vs Tabular only'})
ax.set_title(f'ΔTest R² vs Tabular-only baseline  (green = improvement)', fontsize=11)
ax.set_xlabel('Model', fontsize=10)
ax.set_ylabel('Configuration', fontsize=10)
plt.tight_layout()
plt.savefig('../runs/ablation_delta_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: runs/ablation_delta_heatmap.png')
print()
print(delta_df.to_string())

## 13. Per-Target Breakdown

HoldingTemp and HoldingTime respond differently to each feature stream.

In [ ]:
target_short = ['HoldingTemp', 'HoldingTime']
n_targets = len(target_columns)

fig, axes = plt.subplots(1, n_targets, figsize=(7 * n_targets, 5), sharey=False)
if n_targets == 1:
    axes = [axes]

for t_idx, (ax, t_label) in enumerate(zip(axes, target_short)):
    for i, (cfg_name, color) in enumerate(zip(cfg_names, colors)):
        per_t = [results[cfg_name][m]['per_target'][t_idx] for m in model_names]
        ax.bar(x + i * width, per_t, width, label=cfg_name, color=color, alpha=0.85)
    ax.set_xticks(x + width * (n_cfgs - 1) / 2)
    ax.set_xticklabels(model_names, fontsize=10)
    ax.set_ylabel('Test R²', fontsize=10)
    ax.set_title(f'Per-target: {t_label}', fontsize=11)
    ax.legend(fontsize=7, ncol=2)
    ax.set_ylim(-0.2, 1.05)
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')

plt.suptitle('Per-target Test R² by Feature Configuration', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../runs/ablation_per_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: runs/ablation_per_target.png')

## 14. Feature Importance — Stream Contributions (All-Three Config)

RF trained on the full Tab+Morph+Image matrix.
Bars are colour-coded by stream: purple=image, orange=morph, blue=tabular.

In [ ]:
full_cfg_name = 'Tab + Morph + Image'
if full_cfg_name not in configs:
    full_cfg_name = 'Tabular + Morph' if 'Tabular + Morph' in configs else 'Tabular only'

full_cfg   = configs[full_cfg_name]
feat_names = full_cfg['feat_names']
n_img      = len(img_feature_names)
n_morph    = len(morph_feature_names)

def _stream_color(idx):
    if idx < n_img:              return COLOR_IMG
    if idx < n_img + n_morph:    return COLOR_MORPH
    return COLOR_TAB

rf_imp = RandomForestRegressor(**RF_KW, random_state=42, n_jobs=-1)
rf_imp.fit(full_cfg['X_train'], Y_train)
importances = rf_imp.feature_importances_

top_n   = 30
top_idx = np.argsort(importances)[-top_n:][::-1]
bar_colors = [_stream_color(i) for i in top_idx]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(top_n), importances[top_idx], color=bar_colors, alpha=0.85)
ax.set_xticks(range(top_n))
ax.set_xticklabels(
    [feat_names[i] if i < len(feat_names) else f'f{i}' for i in top_idx],
    rotation=45, ha='right', fontsize=7.5,
)
ax.set_ylabel('RF Feature Importance', fontsize=11)
ax.set_title(f'Top {top_n} Feature Importances — {full_cfg_name}\n'
             '(purple=image, orange=morph, blue=tabular)', fontsize=11)
patches = [
    mpatches.Patch(color=COLOR_IMG,   label=f'Image ({n_img} features)'),
    mpatches.Patch(color=COLOR_MORPH, label=f'Morphological ({n_morph} features)'),
    mpatches.Patch(color=COLOR_TAB,   label=f'Tabular ({len(tab_feature_names)} features)'),
]
ax.legend(handles=patches, fontsize=10)
plt.tight_layout()
plt.savefig('../runs/ablation_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: runs/ablation_feature_importance.png')

img_total   = importances[:n_img].sum()
morph_total = importances[n_img:n_img + n_morph].sum()
tab_total   = importances[n_img + n_morph:].sum()
print(f'\nImportance share ({full_cfg_name}):')
print(f'  Image        : {img_total:.4f}  ({img_total*100:.1f}%)')
print(f'  Morphological: {morph_total:.4f}  ({morph_total*100:.1f}%)')
print(f'  Tabular      : {tab_total:.4f}  ({tab_total*100:.1f}%)')

## 15. Predicted vs Actual — Best Model per Configuration

In [ ]:
fig, axes = plt.subplots(
    len(cfg_names), n_targets,
    figsize=(6 * n_targets, 4.5 * len(cfg_names)),
    squeeze=False,
)

for row, (cfg_name, color) in enumerate(zip(cfg_names, colors)):
    best_model_name = max(results[cfg_name], key=lambda m: results[cfg_name][m]['test_r2'])
    Y_pred = results[cfg_name][best_model_name]['Y_pred']

    for col_idx, t_label in enumerate(target_short):
        ax = axes[row][col_idx]
        y_true = Y_test[:, col_idx]
        y_pred = Y_pred[:, col_idx]

        ax.scatter(y_true, y_pred, alpha=0.7, color=color,
                   edgecolors='white', linewidth=0.5, s=65)
        mn = min(y_true.min(), y_pred.min())
        mx = max(y_true.max(), y_pred.max())
        ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2)
        r2 = r2_score(y_true, y_pred)
        ax.annotate(f'R² = {r2:.4f}', xy=(0.05, 0.92), xycoords='axes fraction',
                    fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_xlabel('Actual', fontsize=9)
        ax.set_ylabel('Predicted', fontsize=9)
        ax.set_title(f'{t_label} — {cfg_name} [{best_model_name}]', fontsize=9)

plt.suptitle('Predicted vs Actual — Best model per feature configuration', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../runs/ablation_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: runs/ablation_pred_vs_actual.png')

## 16. Final Summary

In [ ]:
print('=' * 72)
print('FEATURE STREAM ABLATION — FINAL SUMMARY')
print('=' * 72)
print(f'Dataset  : {len(df_c1)} Cycle 1 samples')
print(f'Backbone : {BACKBONE}')
print(f'Targets  : {target_columns}')
print(f'CV       : RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)')
print(f'Test     : {len(idx_test)} samples (15% hold-out)')
print()

for cfg_name in cfg_names:
    n_feats = configs[cfg_name]['X_train'].shape[1]
    best_by_cv   = max(results[cfg_name], key=lambda m: results[cfg_name][m]['cv_mean'])
    best_by_test = max(results[cfg_name], key=lambda m: results[cfg_name][m]['test_r2'])
    print(f'  ── {cfg_name}  ({n_feats} features) ──')
    print(f'  {"Model":<14} {"CV R²":>12} {"Test R²":>10} {"MAE":>10} {"RMSE":>10}')
    print('  ' + '-' * 60)
    for model_name in model_names:
        m = results[cfg_name][model_name]
        marker = ''
        if model_name == best_by_cv:                              marker += ' ◀ best CV'
        if model_name == best_by_test and model_name != best_by_cv: marker += ' ◀ best test'
        print(f'  {model_name:<14} {m["cv_mean"]:6.4f}±{m["cv_std"]:.4f} '
              f'{m["test_r2"]:10.4f} {m["test_mae"]:10.3f} {m["test_rmse"]:10.3f}{marker}')
    print()

print('  ── ΔTest R² vs Tabular-only baseline ──')
print(f'  {"Configuration":<26}', end='')
for model_name in model_names:
    print(f'{model_name:>10}', end='')
print()
print('  ' + '-' * (26 + 10 * len(model_names)))
for cfg_name in cfg_names:
    if cfg_name == 'Tabular only':
        continue
    print(f'  {cfg_name:<26}', end='')
    for model_name in model_names:
        delta = (results[cfg_name][model_name]['test_r2']
                 - results['Tabular only'][model_name]['test_r2'])
        print(f'{delta:+10.4f}', end='')
    print()
print()
print('=' * 72)